# Demo 02 - the devices reporting

This notebook stands in for the collector that would run next to the hardware: it polls the
devices, turns the vendor's response into a compact reading, and publishes it to the Event Hub.

It is deliberately **not** part of the job. The job is our pipeline, this is the outside world.
If it ran inside the job, every scheduled run would replay the same 53 thousand readings and the
row counts would climb for no reason.

The readings themselves are real: 33 days of telemetry from two QNAP boxes, replayed at speed.
Nothing here is generated except the pace.

In [0]:
%pip install azure-eventhub
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("login", "")
dbutils.widgets.text("target_catalog", "")
dbutils.widgets.text("secret_scope", "")
dbutils.widgets.text("secret_key", "")
dbutils.widgets.text("eventhub_name", "")

login         = dbutils.widgets.get("login")
catalog       = dbutils.widgets.get("target_catalog")
secret_scope  = dbutils.widgets.get("secret_scope")
secret_key    = dbutils.widgets.get("secret_key")
eventhub_name = dbutils.widgets.get("eventhub_name")

assert all([login, catalog, secret_scope, secret_key, eventhub_name])

lab = f"{login}_bronze"
print(f"{catalog}.{lab}.qnap_stats_bronze -> {eventhub_name}")

## What a reading looks like on the wire

The vendor response is a deeply nested blob of about 2 KB per sample. A collector sitting at the
far end of a site link does not ship that, it picks the fields anyone cares about and sends a flat
message of a couple of hundred bytes. Same reason we are doing it here: 53 thousand full blobs is
114 MB through a namespace with one throughput unit, and we do not have that kind of time.

Everything is pulled with `get_json_object` rather than by walking structs. Two fields make that
necessary: `smart_health` is keyed by drive id (`0:1`, `0:2`) and `volumes` by volume name, so the
column names differ per device. Reading them as a MAP and folding over the values works regardless.

Readings where the device failed to return its stats are kept, not filtered. They are a real thing
that happens and the dashboard flags them later.

In [0]:
print(spark.table(f"{catalog}.{lab}.qnap_stats_bronze")
      .select(F.to_json(F.struct("date", "stats")).alias("raw"))
      .first()["raw"][:1000])

In [0]:
raw = F.col("stats")   

def j(path):
    return F.get_json_object(raw, path)


# drives and volumes have per-device key names, so read them as maps and fold over the values
DISK_TEMP = ("array_max(transform(map_values(from_json(get_json_object(stats, '$.smart_health'),"
             " 'MAP<STRING, STRUCT<temp_c INT>>')), d -> d.temp_c))")
VOL_TOTAL = ("aggregate(transform(map_values(from_json(get_json_object(stats, '$.volumes'),"
             " 'MAP<STRING, STRUCT<total_size DOUBLE, free_size DOUBLE>>')), v -> v.total_size),"
             " 0D, (a, b) -> a + coalesce(b, 0D))")
VOL_FREE  = VOL_TOTAL.replace("v.total_size", "v.free_size")

readings = (spark.table(f"{catalog}.{lab}.qnap_stats_bronze")
            .withColumn("vol_total", F.expr(VOL_TOTAL))
            .withColumn("vol_free",  F.expr(VOL_FREE))
            .select(
                j("$.system_stats.system.name").alias("device_id"),
                F.col("date").alias("reading_ts"),
                j("$.system_stats.cpu.usage_percent").cast("double").alias("cpu_usage_pct"),
                F.round(
                    (j("$.system_stats.memory.total").cast("double")
                     - j("$.system_stats.memory.free").cast("double"))
                    / F.nullif(j("$.system_stats.memory.total").cast("double"), F.lit(0.0))
                    * 100, 1).alias("ram_usage_pct"),
                F.expr(DISK_TEMP).alias("disk_temp_c"),
                F.round((F.col("vol_total") - F.col("vol_free"))
                        / F.nullif(F.col("vol_total"), F.lit(0.0)) * 100, 1).alias("disk_used_pct"),
                # system_health is a string normally, but an empty object on a partial reading
                F.when(j("$.system_health").startswith("{"), None)
                 .otherwise(j("$.system_health")).alias("system_health"),
                F.round(F.coalesce(
                    j("$.system_stats.bandwidth.eth0.rx").cast("double"),
                    j("$.bandwidth.eth0.rx").cast("double"),
                    F.lit(0.0)) * 8 / 1_000_000, 1).alias("net_in_mbps"),
            )
            .withColumn("producer", F.lit(login)))

display(readings.limit(5))
print(readings.count(), "readings")

In [0]:
msgs = [r.value for r in
        readings.select(F.to_json(F.struct("*")).alias("value")).collect()]

print(f"{len(msgs)} messages, avg {sum(len(m) for m in msgs) / len(msgs):.0f} bytes")
print(msgs[0])

## Send

Events are packed into batches of about 1 MB, one send call per batch. `batch.add()` raises
`ValueError` when the batch is full and that is the signal to ship it.

No partition key, so the batches land round robin across both partitions and the consumer reads
them in parallel. The namespace has one throughput unit shared with the whole group, so some
throttling is normal and the SDK backs off by itself.

In [0]:
import time

from azure.eventhub import EventHubProducerClient, EventData

conn = dbutils.secrets.get(secret_scope, secret_key)
kwargs = {} if "EntityPath=" in conn else {"eventhub_name": eventhub_name}
producer = EventHubProducerClient.from_connection_string(conn, **kwargs)

sent, batches, t0 = 0, 0, time.time()
with producer:
    batch = producer.create_batch()
    for m in msgs:
        event = EventData(m)
        try:
            batch.add(event)
        except ValueError:
            producer.send_batch(batch)
            sent, batches = sent + len(batch), batches + 1
            batch = producer.create_batch()
            batch.add(event)
    if len(batch):
        producer.send_batch(batch)
        sent, batches = sent + len(batch), batches + 1

took = time.time() - t0
print(f"sent {sent} readings in {batches} batches, {took:.1f}s ({sent / took:.0f}/s)")